# Re‑evaluate Using Saved Byproducts
This notebook reloads the datasets, LoRA weights, and (optionally) the prediction file produced by `med.ipynb`,
then recalculates the NLG metrics.  No training is performed.

In [20]:
!pip install -q nltk rouge-score pycocoevalcap torch

In [21]:
import json, os, sys
from pathlib import Path
from tqdm.notebook import tqdm
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import nltk
from google.colab import drive

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

True

In [22]:
drive.mount('/content/gdrive')
# base_dir = "/content/gdrive/MyDrive/Senior/CS 4782/Project/lora/code"
base_dir = "/content/gdrive/MyDrive/Project/lora/code"
sys.path.append(base_dir)
os.chdir(base_dir)

PROJECT_ROOT = Path(base_dir).resolve().parent
sys.path.append(str(PROJECT_ROOT))

from lora import inject_lora
from generate import generate_batch
import evaluate

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [23]:
# Paths (same as in med.ipynb)
# gptModel = 'gpt2-medium'
gptModel = 'gpt2-large'
DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'

DATASET_FILES = {
    'test': DATA_DIR / f'{gptModel}_testset_w_refs.csv',
}
LORA_WEIGHTS_FILE = RESULTS_DIR / f'{gptModel}_lora_weights.pt'
PREDICTIONS_FILE = RESULTS_DIR / f'{gptModel}_predictions.txt'
RE_EVAL_OUTPUT_FILE = RESULTS_DIR / f'{gptModel}_re_eval_results.json'

# Generation parameters (Table 11)
GEN_BATCH_SIZE = 8

## 1. Load Tokenizer, Model, and LoRA Weights

In [24]:
tokenizer = GPT2Tokenizer.from_pretrained(gptModel)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

model = GPT2LMHeadModel.from_pretrained(gptModel)
model = inject_lora(model, rank=4, alpha=32)

# Load saved LoRA weights (only trainable parameters)
state_dict = torch.load(LORA_WEIGHTS_FILE, map_location='cpu')
model.load_state_dict(state_dict, strict=False)
model.eval()
print(f"Loaded LoRA weights from {LORA_WEIGHTS_FILE}")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.25G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/436 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-large
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...35}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded LoRA weights from /content/gdrive/.shortcut-targets-by-id/1X7tG3A2pprArWBo7iRD_4bw2iX8rrXdY/Project/lora/results/gpt2-large_lora_weights.pt


In [25]:
DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# gptModel = 'gpt2-medium'
gptModel = 'gpt2-large'

DATASET_FILES = {
    'train': DATA_DIR / f'{gptModel}_trainset.csv',
    'validation': DATA_DIR / f'{gptModel}_devset.csv',
    'test': DATA_DIR / f'{gptModel}_testset_w_refs.csv',
}
LORA_WEIGHTS_FILE = RESULTS_DIR / f'{gptModel}_lora_weights.pt'
PREDICTIONS_FILE = RESULTS_DIR / f'{gptModel}_predictions.txt'
OUTPUT_FILE = RESULTS_DIR / f'{gptModel}_results.json'

# Table 11
BEAM_SIZE = 10
LENGTH_PENALTY = 0.9
NO_REPEAT_NGRAM = 4
MAX_NEW_TOKENS = 128
GEN_BATCH_SIZE = 8


In [26]:
from datasets import Dataset, DatasetDict
import pandas as pd

def load_e2e(gptModel):
    base = "https://raw.githubusercontent.com/tuetschek/e2e-dataset/master/"

    def rename(df):
        return df.rename(columns={"mr": "input", "ref": "label"})

    source_files = {
        'train': 'trainset.csv',
        'validation': 'devset.csv',
        'test': 'testset_w_refs.csv',
    }

    for split, filename in source_files.items():
        csv_path = DATASET_FILES[split]
        if not csv_path.exists():
            pd.read_csv(base + filename).to_csv(csv_path, index=False)

    return DatasetDict({
        "train": Dataset.from_pandas(rename(pd.read_csv(DATASET_FILES['train']))),
        "validation": Dataset.from_pandas(rename(pd.read_csv(DATASET_FILES['validation']))),
        "test": Dataset.from_pandas(rename(pd.read_csv(DATASET_FILES['test']))),
    })

dataset = load_e2e(gptModel)
print(f"Dataset files stored in: {DATA_DIR}")

Dataset files stored in: /content/gdrive/.shortcut-targets-by-id/1X7tG3A2pprArWBo7iRD_4bw2iX8rrXdY/Project/lora/data


In [27]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer.padding_side = 'left'

from collections import OrderedDict

ref_groups = OrderedDict()
for inp, label in zip(dataset['test']['input'], dataset['test']['label']):
    if inp not in ref_groups:
        ref_groups[inp] = []
    ref_groups[inp].append(label)

# prompts = list(ref_groups.keys())
prompts = [f"Input: {inp}\nOutput:" for inp in ref_groups.keys()]
references = list(ref_groups.values())

print(f'Unique MRs (prompts to generate): {len(prompts)}')
print(f'Total reference texts:            {sum(len(r) for r in references)}')
print(f'Average refs per MR:              {sum(len(r) for r in references)/len(prompts):.1f}')
print(f'Sample prompt: {prompts[0][:80]}...')


Unique MRs (prompts to generate): 630
Total reference texts:            4693
Average refs per MR:              7.4
Sample prompt: Input: name[Blue Spice], eatType[coffee shop], area[city centre]
Output:...


In [28]:
model.to(device)

# Run generation over all test prompts
all_predictions = []
for i in tqdm(range(0, len(prompts), GEN_BATCH_SIZE), desc='Generating'):
    tokenizer.padding_side = "left"
    raw_batch = prompts[i : i + GEN_BATCH_SIZE]
    batch = tokenizer(
        raw_batch,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    )
    batch = {k: v.to(device) for k, v in batch.items()}
    all_predictions.extend(generate_batch(model, tokenizer, batch, device))

print(f'\nGenerated {len(all_predictions)} predictions')
print('\nSample outputs:')
for i in range(min(3, len(all_predictions))):
    print(f'  [{i}] Prompt: {prompts[i][:60]}...')
    print(f'       Output: {all_predictions[i]}')
    print()

Generating:   0%|          | 0/79 [00:00<?, ?it/s]


Generated 630 predictions

Sample outputs:
  [0] Prompt: Input: name[Blue Spice], eatType[coffee shop], area[city cen...
       Output: The city is a coffee shop that is a good place to go.

  [1] Prompt: Input: name[Blue Spice], eatType[coffee shop], area[riversid...
       Output: a lot of food and a lot.

  [2] Prompt: Input: name[Blue Spice], eatType[coffee shop], customer rati...
       Output: The Crown of Crown of Crown is the Crown of Crown.



In [29]:
# Save predictions
with open(PREDICTIONS_FILE, 'w', encoding='utf-8') as f:
    for pred in all_predictions:
        f.write(pred + '\n')

print(f'Saved predictions to: {PREDICTIONS_FILE}')


Saved predictions to: /content/gdrive/.shortcut-targets-by-id/1X7tG3A2pprArWBo7iRD_4bw2iX8rrXdY/Project/lora/results/gpt2-large_predictions.txt


## 2. Prepare Reference Groups (from Test Set CSV)

In [30]:
import pandas as pd
from collections import OrderedDict

# Load the test CSV (one row per reference)
test_df = pd.read_csv(DATASET_FILES['test']).rename(columns={"mr": "input", "ref": "label"})

# Group references by prompt (MR)
ref_groups = OrderedDict()
for inp, label in zip(test_df['input'], test_df['label']):
    if inp not in ref_groups:
        ref_groups[inp] = []
    ref_groups[inp].append(label)

prompts = list(ref_groups.keys())
references = list(ref_groups.values())

print(f"Unique MRs: {len(prompts)}")
print(f"Total references: {sum(len(r) for r in references)}")
print(f"Avg refs per MR: {sum(len(r) for r in references)/len(prompts):.1f}")

Unique MRs: 630
Total references: 4693
Avg refs per MR: 7.4


## 3. Load or Generate Predictions

In [31]:
if PREDICTIONS_FILE.exists():
    with open(PREDICTIONS_FILE, 'r', encoding='utf-8') as f:
        all_predictions = [line.strip() for line in f]
    print(f"Loaded {len(all_predictions)} predictions from {PREDICTIONS_FILE}")
else:
    print("Predictions file not found – generating from model ...")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    all_predictions = []
    for i in tqdm(range(0, len(prompts), GEN_BATCH_SIZE), desc='Generating'):
        raw_batch = prompts[i : i + GEN_BATCH_SIZE]
        batch = tokenizer(raw_batch, return_tensors="pt", padding=True,
                          truncation=True, max_length=512)
        batch = {k: v.tolist() for k, v in batch.items()}
        all_predictions.extend(generate_batch(model, tokenizer, batch, device))

    # Save for future reuse
    with open(PREDICTIONS_FILE, 'w', encoding='utf-8') as f:
        for pred in all_predictions:
            f.write(pred + '\n')
    print(f"Generated and saved {len(all_predictions)} predictions to {PREDICTIONS_FILE}")

Loaded 630 predictions from /content/gdrive/.shortcut-targets-by-id/1X7tG3A2pprArWBo7iRD_4bw2iX8rrXdY/Project/lora/results/gpt2-large_predictions.txt


## 4. Compute NLG Metrics

In [32]:
print("Computing metrics...")
bleu = evaluate.compute_bleu(all_predictions, references)
nist = evaluate.compute_nist(all_predictions, references)
meteor = evaluate.compute_meteor(all_predictions, references)
rouge_l = evaluate.compute_rouge_l(all_predictions, references)
cider = evaluate.compute_cider(all_predictions, references)

print(f"BLEU   : {bleu}")
print(f"NIST   : {nist}")
print(f"METEOR : {meteor}")
print(f"ROUGE-L: {rouge_l}")
print(f"CIDEr  : {cider}")

Computing metrics...
BLEU   : 29.36
NIST   : 0.9916
METEOR : 25.13
ROUGE-L: 43.26
CIDEr  : 0.3678


## 5. Save Results

In [33]:
results = {
    'num_examples': len(all_predictions),
    'predictions_file': str(PREDICTIONS_FILE),
    'lora_weights_file': str(LORA_WEIGHTS_FILE),
    'BLEU': bleu,
    'NIST': nist,
    'METEOR': meteor,
    'ROUGE-L': rouge_l,
    'CIDEr': cider,
}

with open(RE_EVAL_OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)

print(f"Re‑evaluation results saved to {RE_EVAL_OUTPUT_FILE}")

Re‑evaluation results saved to /content/gdrive/.shortcut-targets-by-id/1X7tG3A2pprArWBo7iRD_4bw2iX8rrXdY/Project/lora/results/gpt2-large_re_eval_results.json
